# ZEST — Kaggle GPU smoke-test notebook

Runs the full 5-stage ZEST pipeline end-to-end on a **tiny ESD subset** to produce at least one
emotion-converted `.wav`. It reuses the shipped `f0.pickle` + HuBERT-token manifests, patches every
hardcoded path at runtime, and fixes the `pickle5` / `torch.load` portability bugs.

**Before running:** Notebook settings → Accelerator = **GPU**, Internet = **ON**.
Edit `REPO_SRC` and `ESD_WAV_DIR` in Cell 1 to match your added inputs. Then run cells top-to-bottom.


In [ ]:
# ============ Cell 1: setup, paths, env ============
import os, sys, subprocess, shutil, ast, json
from pathlib import Path

# ---- EDIT THESE TO MATCH YOUR SETUP ----
REPO_URL    = "https://github.com/vanshpatil16/zest"   # ZEST repo, cloned at runtime (repo must be PUBLIC, or use a token URL)
ESD_WAV_DIR = "/kaggle/input/esd"         # root that contains ESD English .wav files (searched recursively)
# -----------------------------------------

REPO = "/kaggle/working/ZEST"
CODE = REPO + "/code"
WORK = "/kaggle/working/zest"

# Clone the repo into the writable working area (Internet must be ON in Notebook options)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)

P = {
    "DATA":           WORK + "/data",
    "TRAIN_DIR":      WORK + "/data/train",
    "VAL_DIR":        WORK + "/data/val",
    "TEST_DIR":       WORK + "/data/test",
    "XVECTOR_DIR":    WORK + "/x_vectors",
    "EASE_EMB_DIR":   WORK + "/EASE_embeddings",
    "F0_CONTOUR_DIR": WORK + "/f0_contours",
    "WAV2VEC_DIR":    WORK + "/wav2vec_feats",
    "PRED_DSDT_DIR":  WORK + "/pred_DSDT_f0",
    "CKPT_DIR":       WORK + "/checkpoints",
    "OUTPUT_DIR":     WORK + "/converted",
}
for d in P.values():
    os.makedirs(d, exist_ok=True)

# subset manifests we will write in Cell 4 (audio paths rewritten to the copied wavs)
P["TRAIN_MANIFEST"] = WORK + "/train_subset.txt"
P["VAL_MANIFEST"]   = WORK + "/val_subset.txt"
P["TEST_MANIFEST"]  = WORK + "/test_subset.txt"
# shipped artifacts reused as-is
P["F0_PICKLE"]      = CODE + "/f0.pickle"
F0_STATS            = CODE + "/esd_f0_stats.pth"
HIFIGAN_CONFIG      = WORK + "/hifigan_kaggle.json"

# smoke-size knobs (raise these for a fuller run)
P["EASE_EPOCHS"]    = "3"
P["F0_EPOCHS"]      = "2"
P["HIFIGAN_STEPS"]  = "200"
UTTS_PER_BUCKET     = 2          # train wavs per (speaker, emotion)
VAL_UTTS            = 1          # val/test wavs per (speaker, emotion)

# environment passed to every child process
ENV = dict(os.environ)
ENV.update(P)
ENV["PYTHONPATH"] = CODE + os.pathsep + ENV.get("PYTHONPATH", "")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - enable the accelerator!"))
print("repo cloned (code/ present):", os.path.isdir(CODE), "| ESD_WAV_DIR exists:", os.path.isdir(ESD_WAV_DIR))

In [ ]:
# ============ Cell 2: install deps (drop pickle5 - it won't build on Kaggle's Python) ============
req = Path(REPO + "/requirements.txt")
kept = [l for l in req.read_text().splitlines() if "pickle5" not in l]
req.write_text("\n".join(kept) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
print("dependencies installed (pickle5 removed; stdlib pickle reads protocol 5)")

In [ ]:
# ============ Cell 3: patch hardcoded paths + portability bugs in the working copy ============
def patch(rel, repls):
    f = Path(CODE) / rel
    s = f.read_text()
    for old, new in repls:
        if old not in s:
            raise AssertionError("PATCH MISS in %s for:\n%s" % (rel, old))
        s = s.replace(old, new)
    f.write_text(s)
    print("patched", rel)

patch("F0_predictor/config.py", [
    ('train_datasets = {"ESD":"/home/soumyad/emoconv/ESD/train"}',
     'import os\ntrain_datasets = {"ESD": os.environ["TRAIN_DIR"]}'),
    ('val_datasets = {"ESD":"/home/soumyad/emoconv/ESD/val"}',
     'val_datasets = {"ESD": os.environ["VAL_DIR"]}'),
    ('test_datasets = {"ESD":"/home/soumyad/emoconv/ESD/test"}',
     'test_datasets = {"ESD": os.environ["TEST_DIR"]}'),
    ('train_tokens_orig = {"ESD":"/ZEST/code/train_esd.txt"}',
     'train_tokens_orig = {"ESD": os.environ["TRAIN_MANIFEST"]}'),
    ('val_tokens_orig = {"ESD":"/ZEST/code/val_esd.txt"}',
     'val_tokens_orig = {"ESD": os.environ["VAL_MANIFEST"]}'),
    ('test_tokens_orig = {"ESD":"/ZEST/code/test_esd.txt"}',
     'test_tokens_orig = {"ESD": os.environ["TEST_MANIFEST"]}'),
    ('f0_file = "ZEST/code/f0.pickle"',
     'f0_file = os.environ["F0_PICKLE"]'),
])

patch("EASE/get_speaker_embedding.py", [
    ('folder = "/folder/to/wav_files"',          'folder = os.environ["EASE_WAV_DIR"]'),
    ('target_folder = "/folder/to/store/x-vectors"', 'target_folder = os.environ["XVECTOR_DIR"]'),
])

patch("EASE/speaker_classifier.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ('    speaker_folder = "/folder/to/x-vectors"', '    speaker_folder = os.environ["XVECTOR_DIR"]'),
    ('        folder = "/folder/to/train/audio/files"',      '        folder = os.environ["TRAIN_DIR"]'),
    ('        folder = "/folder/to/validation/audio/files"', '        folder = os.environ["VAL_DIR"]'),
    ('        folder = "/folder/to/test/audio/files"',       '        folder = os.environ["TEST_DIR"]'),
    ('    for e in range(10):', '    for e in range(int(os.environ.get("EASE_EPOCHS", "3"))):'),
    ("model = torch.load('EASE.pth', map_location=device)",
     "model = torch.load('EASE.pth', map_location=device, weights_only=False)"),
    ('os.makedirs("EASE_embeddings", exist_ok=True)', 'os.makedirs(os.environ["EASE_EMB_DIR"], exist_ok=True)'),
    ('np.save(os.path.join("EASE_embeddings", target_file_name)',
     'np.save(os.path.join(os.environ["EASE_EMB_DIR"], target_file_name)'),
])

patch("F0_predictor/pitch_attention_adv.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ('np.load(os.path.join("/folder/to/EASE/embeddings", file_name.replace(".wav", ".npy")))',
     'np.load(os.path.join(os.environ["EASE_EMB_DIR"], file_name.replace(".wav", ".npy")))'),
    ('        folder = "/folder/to/train/audio/files"',      '        folder = os.environ["TRAIN_DIR"]'),
    ('        folder = "/folder/to/validation/audio/files"', '        folder = os.environ["VAL_DIR"]'),
    ('        folder = "/folder/to/test/audio/files"',       '        folder = os.environ["TEST_DIR"]'),
    ('    for e in range(500):', '    for e in range(int(os.environ.get("F0_EPOCHS", "2"))):'),
])

patch("F0_predictor/pitch_inference.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ("model = torch.load('f0_predictor.pth', map_location=device)",
     "model = torch.load('f0_predictor.pth', map_location=device, weights_only=False)"),
    ('os.makedirs("f0_contours", exist_ok=True)', 'os.makedirs(os.environ["F0_CONTOUR_DIR"], exist_ok=True)'),
    ('np.save(os.path.join("f0_contours", target_file_name)',
     'np.save(os.path.join(os.environ["F0_CONTOUR_DIR"], target_file_name)'),
])

patch("F0_predictor/get_wav2vec_feats.py", [
    ('from config import hparams, f0_stats', 'from config import hparams'),
    ('import pickle5 as pickle', 'import pickle'),
    ("model = torch.load('f0_predictor.pth', map_location=device)",
     "model = torch.load('f0_predictor.pth', map_location=device, weights_only=False)"),
    ('    wav2vec_feats_folder = "wav2vec_feats"', '    wav2vec_feats_folder = os.environ["WAV2VEC_DIR"]'),
])

patch("F0_predictor/pitch_convert.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ("model = torch.load('f0_predictor.pth', map_location=device)",
     "model = torch.load('f0_predictor.pth', map_location=device, weights_only=False)"),
    ('os.makedirs("pred_DSDT_f0", exist_ok=True)', 'os.makedirs(os.environ["PRED_DSDT_DIR"], exist_ok=True)'),
    ('np.save(os.path.join("pred_DSDT_f0", final_name)', 'np.save(os.path.join(os.environ["PRED_DSDT_DIR"], final_name)'),
    ('''    sources = ["0011_000021.wav", "0012_000022.wav", "0013_000025.wav",
               "0014_000032.wav", "0015_000034.wav", "0016_000035.wav",
               "0017_000038.wav", "0018_000043.wav", "0019_000023.wav",
               "0020_000047.wav"]''',
     '    sources = sorted(f for f in os.listdir(os.environ["TEST_DIR"]) if f.endswith(".wav") and int(f[5:11]) <= 350)'),
])

patch("HiFi-GAN/dataset.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ('feats[\'spkr\'] = np.load("/ZEST/code/EASE/EASE_embeddings/" + emo_file_name)',
     'feats[\'spkr\'] = np.load(os.environ["EASE_EMB_DIR"] + "/" + emo_file_name)'),
])

patch("HiFi-GAN/inference.py", [
    ('reference_files = os.listdir("/folder/to/ESD/test/wavs")',
     'reference_files = os.listdir(os.environ["TEST_DIR"])'),
    ('emo_embed = np.load("/ZEST/code/F0_predictor/wav2vec_feats/" + filename.replace(".wav", ".npy"))',
     'emo_embed = np.load(os.environ["WAV2VEC_DIR"] + "/" + filename.replace(".wav", ".npy"))'),
    ('f0 = np.load("/ZEST/code/F0_predictor/pred_DSDT_f0" + fname_out_name + filename.replace(".wav", ".npy"))',
     'f0 = np.load(os.environ["PRED_DSDT_DIR"] + "/" + fname_out_name + filename.replace(".wav", ".npy"))'),
])

print("\nAll patches applied.")

In [ ]:
# ============ Cell 4: Stage 0 - build a tiny subset + rewrite manifest audio paths ============
# Reuses the shipped *_esd.txt (HuBERT tokens) and f0.pickle. Only the wav FILES come from ESD.
EMO_BOUNDS = [(0, 350), (351, 700), (701, 1050), (1051, 1400), (1401, 10**9)]
def bucket(fid):
    for i, (lo, hi) in enumerate(EMO_BOUNDS):
        if lo <= fid <= hi:
            return i
    return 4

def read_manifest(path):
    out = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line:
            out.append(ast.literal_eval(line))
    return out

def base(rec):
    return rec["audio"].split("/")[-1].split("\\")[-1]

# index every ESD wav by basename (first match wins)
wav_index = {}
for root, _, files in os.walk(ESD_WAV_DIR):
    for fn in files:
        if fn.endswith(".wav"):
            wav_index.setdefault(fn, os.path.join(root, fn))
print("Indexed", len(wav_index), "ESD wavs under", ESD_WAV_DIR)

def build_split(full_manifest, dest_dir, manifest_out, per_bucket):
    recs = read_manifest(full_manifest)
    from collections import defaultdict
    grouped = defaultdict(list)
    for r in recs:
        bn = base(r)
        if bn in wav_index:                      # only keep wavs that actually exist in the ESD input
            grouped[(bn[:4], bucket(int(bn[5:11])))].append(r)
    written = 0
    lines = []
    for key in sorted(grouped):
        for r in sorted(grouped[key], key=base)[:per_bucket]:
            bn = base(r)
            dst = os.path.join(dest_dir, bn)
            if not os.path.exists(dst):
                shutil.copy2(wav_index[bn], dst)
            r2 = dict(r)
            r2["audio"] = dst.replace("\\", "/")  # rewrite to the copied location
            lines.append(str(r2))
            written += 1
    Path(manifest_out).write_text("\n".join(lines) + "\n")
    return written

n_tr = build_split(CODE + "/train_esd.txt", P["TRAIN_DIR"], P["TRAIN_MANIFEST"], UTTS_PER_BUCKET)
n_va = build_split(CODE + "/val_esd.txt",   P["VAL_DIR"],   P["VAL_MANIFEST"],   VAL_UTTS)
n_te = build_split(CODE + "/test_esd.txt",  P["TEST_DIR"],  P["TEST_MANIFEST"],  VAL_UTTS)
print(f"subset wavs -> train {n_tr}, val {n_va}, test {n_te}")
if min(n_tr, n_va, n_te) == 0:
    print("!! A split is EMPTY. Your ESD_WAV_DIR basenames don't match the manifests. Fix ESD_WAV_DIR.")

# write a Kaggle HiFi-GAN config from the shipped template
cfg = json.loads(Path(CODE + "/HiFi-GAN/hubert_alladv.json").read_text())
cfg["input_training_file"]   = P["TRAIN_MANIFEST"]
cfg["input_validation_file"] = P["VAL_MANIFEST"]
cfg["f0_stats"]   = F0_STATS
cfg["num_gpus"]   = 0
cfg["batch_size"] = 4
cfg["num_workers"] = 0
Path(HIFIGAN_CONFIG).write_text(json.dumps(cfg, indent=2))
print("HiFi-GAN config ->", HIFIGAN_CONFIG)

In [ ]:
# ============ Cell 5: Stage 1 - EASE (x-vectors -> adversarial speaker encoder -> embeddings) ============
EASE = CODE + "/EASE"
def run(cmd, cwd, env=None):
    print(">>", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], cwd=cwd, env=(env or ENV), check=True)

# x-vectors for all three splits (speaker_classifier reads them all from XVECTOR_DIR)
for folder in (P["TRAIN_DIR"], P["VAL_DIR"], P["TEST_DIR"]):
    e = dict(ENV); e["EASE_WAV_DIR"] = folder
    run([sys.executable, EASE + "/get_speaker_embedding.py"], EASE, e)
print("x-vectors:", len(list(Path(P["XVECTOR_DIR"]).glob("*.npy"))))

run([sys.executable, EASE + "/speaker_classifier.py"], EASE)
print("EASE embeddings:", len(list(Path(P["EASE_EMB_DIR"]).glob("*.npy"))))

In [ ]:
# ============ Cell 6: Stage 2 - F0 predictor (train -> contours -> SACE wav2vec feats) ============
F0 = CODE + "/F0_predictor"
run([sys.executable, F0 + "/pitch_attention_adv.py"], F0)   # writes f0_predictor.pth into F0/
run([sys.executable, F0 + "/pitch_inference.py"], F0)       # -> F0_CONTOUR_DIR
run([sys.executable, F0 + "/get_wav2vec_feats.py"], F0)     # -> WAV2VEC_DIR
print("f0_contours:", len(list(Path(P["F0_CONTOUR_DIR"]).glob("*.npy"))),
      "| wav2vec_feats:", len(list(Path(P["WAV2VEC_DIR"]).glob("*.npy"))))

In [ ]:
# ============ Cell 7: Stage 3 - train HiFi-GAN (smoke: a few hundred steps) ============
HG = CODE + "/HiFi-GAN"
run([sys.executable, HG + "/train.py",
     "--checkpoint_path", P["CKPT_DIR"],
     "--config", HIFIGAN_CONFIG,
     "--pitch_folder", P["F0_CONTOUR_DIR"] + "/",
     "--emo_folder",   P["WAV2VEC_DIR"] + "/",
     "--training_steps", P["HIFIGAN_STEPS"],
     "--checkpoint_interval", P["HIFIGAN_STEPS"]], HG)
print("checkpoints:", [p.name for p in Path(P["CKPT_DIR"]).glob("g_*")])

In [ ]:
# ============ Cell 8: Stage 4 - convert F0 (DSDT) + HiFi-GAN inference -> converted wav ============
F0 = CODE + "/F0_predictor"; HG = CODE + "/HiFi-GAN"
run([sys.executable, F0 + "/pitch_convert.py"], F0)
print("pred_DSDT_f0:", len(list(Path(P["PRED_DSDT_DIR"]).glob("*.npy"))))

run([sys.executable, HG + "/inference.py", "--convert",
     "--checkpoint_file", P["CKPT_DIR"],
     "--output_dir",      P["OUTPUT_DIR"],
     "--emo_folder",      P["WAV2VEC_DIR"] + "/",
     "--pitch_folder",    P["F0_CONTOUR_DIR"] + "/",
     "--f0-stats",        F0_STATS,
     "--input_code_file", P["TEST_MANIFEST"]], HG)

wavs = sorted(Path(P["OUTPUT_DIR"]).glob("*.wav"))
print("CONVERTED WAVS:", [w.name for w in wavs][:10])
import IPython.display as ipd
if wavs:
    ipd.display(ipd.Audio(str(wavs[0])))
else:
    print("No converted wavs. The test subset may lack a valid neutral-source / emotional-reference pair.")
    print("Fix: raise VAL_UTTS in Cell 1 (e.g. 3) so more speakers/emotions land in the test split, then re-run Cells 4 and 8.")

In [ ]:
# ============ Cell 9: run report ============
for label, d, pat in [("subset train", P["TRAIN_DIR"], "*.wav"), ("subset test", P["TEST_DIR"], "*.wav"),
                      ("x-vectors", P["XVECTOR_DIR"], "*.npy"), ("EASE emb", P["EASE_EMB_DIR"], "*.npy"),
                      ("f0_contours", P["F0_CONTOUR_DIR"], "*.npy"), ("wav2vec_feats", P["WAV2VEC_DIR"], "*.npy"),
                      ("pred_DSDT_f0", P["PRED_DSDT_DIR"], "*.npy"), ("checkpoints", P["CKPT_DIR"], "g_*"),
                      ("CONVERTED wavs", P["OUTPUT_DIR"], "*.wav")]:
    print(f"{label:16s}: {len(list(Path(d).glob(pat)))}")